# Portfolio Backtest & Risk Diagnostics

Full walk-forward backtest summary plus portfolio-risk diagnostics from the persisted `wf_portfolio_*` tables.

Regenerate results with:

```bash
uv run python research/portfolio/walk_forward.py
```

This notebook is backward-compatible with older saved runs. Per-bar execution/risk sections require a run saved by the newer portfolio code.

In [ ]:
import os, sys
from pathlib import Path

if not os.getcwd().endswith('crypto'):
    os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from scipy.stats import norm

from dbutil import load_data, get_table_columns
from config import BARS_PER_DAY

PPY = BARS_PER_DAY * 365
ANN = np.sqrt(PPY)

%load_ext autoreload
%autoreload 2

## Load Tables

In [ ]:
returns = load_data('wf_portfolio_returns')
windows = load_data('wf_portfolio_windows')
exposures = load_data('wf_portfolio_exposures')

assert returns is not None and not returns.empty, 'No wf_portfolio_returns rows. Run research/portfolio/walk_forward.py first.'
assert windows is not None and not windows.empty, 'No wf_portfolio_windows rows. Run research/portfolio/walk_forward.py first.'

returns['timestamp'] = pd.to_datetime(returns['timestamp'])
# Walk-forward test windows OVERLAP by a few days at each boundary: a window's
# test span is (train_start + 6 months + 30 days) but the next starts 30 days
# later, and DateOffset(months=...) is not commutative with Timedelta(days=...),
# so adjacent spans share ~1-3 days. Each shared bar is therefore saved twice
# (once per window). Left in, cumprod double-counts ~1700 bars - including the
# worst Feb-2024 days at the W0/W1 boundary - inflating drawdown and deflating
# Sharpe. Keep the EARLIER window's copy (it traded that bar in causal sequence;
# the later window re-simulated it starting from a carried, look-ahead book).
if 'window' in returns.columns:
    _n0 = len(returns)
    returns = returns.sort_values(['window', 'timestamp']).drop_duplicates('timestamp', keep='first')
    _ndup = _n0 - len(returns)
    if _ndup:
        print(f"de-overlapped returns: dropped {_ndup:,} duplicate window-boundary bars")
returns = returns.set_index('timestamp').sort_index()

windows = windows.sort_values('window_idx').reset_index(drop=True)
windows['train_start'] = pd.to_datetime(windows['train_start'])
windows['train_end'] = pd.to_datetime(windows['train_end'])
windows['test_end'] = pd.to_datetime(windows['test_end'])
windows['test_start'] = windows['train_end']

risk_cols = ['gross_return', 'turnover', 'gross_exposure', 'net_exposure',
             'mkt_exposure', 'size_exposure', 'n_positions']
has_risk_cols = all(c in returns.columns for c in risk_cols)

if exposures is not None and not exposures.empty:
    exposures['timestamp'] = pd.to_datetime(exposures['timestamp'])
    if 'window' in exposures.columns:
        exposures = exposures.sort_values(['window', 'timestamp']).drop_duplicates('timestamp', keep='first')
    exposures = exposures.set_index('timestamp').sort_index()

print(f"return bars : {len(returns):,} ({returns.index.min()} -> {returns.index.max()})")
print(f"windows     : {len(windows):,}")
print(f"run timestamp: {returns['run_timestamp'].iloc[-1] if 'run_timestamp' in returns.columns else 'n/a'}")
print(f"per-bar risk diagnostics available: {has_risk_cols}")
print('return columns:', ', '.join(returns.columns))

## Full Backtest Summary

In [ ]:
def max_drawdown(r):
    wealth = (1 + r.fillna(0)).cumprod()
    return (wealth / wealth.cummax() - 1).min()

def perf_stats(r, ppy=PPY):
    r = r.dropna()
    if r.empty:
        return pd.Series(dtype=float)
    total = (1 + r).prod() - 1
    years = len(r) / ppy
    ann_ret_geo = (1 + total) ** (1 / years) - 1 if years > 0 and total > -1 else np.nan
    ann_ret_arith = r.mean() * ppy
    ann_vol = r.std() * np.sqrt(ppy)
    downside = r[r < 0].std() * np.sqrt(ppy)
    dd = max_drawdown(r)
    return pd.Series({
        'bars': len(r),
        'years': years,
        'total_return_%': total * 100,
        'ann_return_geo_%': ann_ret_geo * 100,
        'ann_return_arith_%': ann_ret_arith * 100,
        'ann_vol_%': ann_vol * 100,
        'sharpe': ann_ret_arith / ann_vol if ann_vol > 0 else np.nan,
        'sortino': ann_ret_arith / downside if downside > 0 else np.nan,
        'max_drawdown_%': dd * 100,
        'calmar': ann_ret_geo / abs(dd) if dd < 0 and pd.notna(ann_ret_geo) else np.nan,
        'hit_rate_%': (r > 0).mean() * 100,
        'mean_bps_bar': r.mean() * 10000,
        'std_bps_bar': r.std() * 10000,
        'skew': r.skew(),
        'excess_kurtosis': r.kurtosis(),
    })

summary = pd.DataFrame({'net': perf_stats(returns['net_return'])}).T
if 'net_return_lag1' in returns.columns:
    summary.loc['net_1bar_lag'] = perf_stats(returns['net_return_lag1'])
if has_risk_cols:
    summary.loc['gross_before_costs'] = perf_stats(returns['gross_return'])
    cost = returns['gross_return'] - returns['net_return']
    summary.loc['cost_drag'] = perf_stats(-cost)

summary.round(3)

## Equity, Drawdown, Participation, Turnover

In [ ]:
from config import get as _cfg

r = returns['net_return'].fillna(0)
cum = (1 + r).cumprod() - 1
dd = (1 + cum) / (1 + cum).cummax() - 1

# Stacked panels share the x axis for alignment, but each panel gets its OWN
# legend anchored to that panel (no more single merged legend for everything).
has_gross = has_risk_cols and 'gross_exposure' in returns.columns
has_part = 'participation_max' in returns.columns
has_turnover = 'turnover' in returns.columns
part_cap = _cfg('portfolio.participation.max_participation')   # per-symbol, % of volume
book_cap = _cfg('portfolio.participation.max_book_turnover')   # whole book, % of book

panels = ['Cumulative return', 'Drawdown']
if has_gross:
    panels.append('Gross exposure')
if has_part:
    panels.append('Participation / bar')
if has_turnover:
    panels.append('Turnover / bar')
nrows = len(panels)
row = {name: i + 1 for i, name in enumerate(panels)}
legname = lambda i: 'legend' if i == 1 else f'legend{i}'

fig = make_subplots(rows=nrows, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=panels)

def add(title, trace):
    i = row[title]
    trace.update(legend=legname(i))          # route this trace to its panel's legend
    fig.add_trace(trace, row=i, col=1)

add('Cumulative return',
    go.Scatter(x=returns.index, y=cum * 100, name='net', line=dict(color='navy')))
if 'net_return_lag1' in returns.columns:
    lag_cum = (1 + returns['net_return_lag1'].fillna(0)).cumprod() - 1
    add('Cumulative return',
        go.Scatter(x=returns.index, y=lag_cum * 100, name='1-bar lag',
                   line=dict(color='lightsteelblue', dash='dot')))
add('Drawdown',
    go.Scatter(x=returns.index, y=dd * 100, name='drawdown', fill='tozeroy',
               line=dict(color='firebrick')))
if has_gross:
    add('Gross exposure',
        go.Scatter(x=returns.index, y=returns['gross_exposure'], name='gross',
                   fill='tozeroy', line=dict(color='darkgreen')))
if has_part:
    # Per-symbol participation: each bar's trade as a fraction of the traded
    # name's OWN bar volume. Peak (the binding name) and mean across names.
    add('Participation / bar',
        go.Scatter(x=returns.index, y=returns['participation_max'], name='max name',
                   line=dict(color='purple')))
    add('Participation / bar',
        go.Scatter(x=returns.index, y=returns['participation_mean'], name='mean',
                   line=dict(color='plum')))
    if part_cap is not None:
        fig.add_hline(y=part_cap, line_dash='dash', line_color='gray',
                      annotation_text=f'cap {part_cap*100:g}% of volume',
                      annotation_position='top left',
                      row=row['Participation / bar'], col=1)
if has_turnover:
    # Whole-book turnover: sum of |dw| across all names (+ forced closes of names
    # leaving the universe) as a fraction of the book.
    add('Turnover / bar',
        go.Scatter(x=returns.index, y=returns['turnover'], name='turnover',
                   fill='tozeroy', line=dict(color='orange')))
    if book_cap is not None:
        fig.add_hline(y=book_cap, line_dash='dash', line_color='gray',
                      annotation_text=f'cap {book_cap*100:g}% of book',
                      annotation_position='top left',
                      row=row['Turnover / bar'], col=1)

fig.update_layout(height=240 * nrows, hovermode='x unified', template='plotly_white')
# Pin each legend box to the top-right of its own panel's y-domain.
for i in range(1, nrows + 1):
    b, t = fig.layout['yaxis' if i == 1 else f'yaxis{i}'].domain
    fig.update_layout({legname(i): dict(
        xref='paper', x=1.005, xanchor='left',
        yref='paper', y=t, yanchor='top',
        bgcolor='rgba(255,255,255,0.6)', bordercolor='lightgray', borderwidth=1,
        font=dict(size=10))})
fig.show()

## Signal-Level Edge & First-Window Diagnostics

The net equity mixes signal quality with the cold-start gross ramp. Normalizing each bar's PnL by its gross exposure isolates the **signal edge**: if the early drop were just low exposure while ramping, the gross-normalized line would be flat early. It is not — the first window's signal edge is genuinely negative (the late-Feb-2024 melt-up), and the loss is a **fat negative tail** (worst 1% of bars), not a broad grind. This is a real out-of-sample loss in the most fragile window, not an artifact.

In [ ]:
# Signal-level edge vs exposure ramp. Net equity mixes signal quality with the
# cold-start gross ramp; dividing each bar's PnL by its gross exposure isolates
# the signal edge. If the early drop were merely "low gross while ramping", the
# purple line would be flat early - it is not, so the first window's signals had
# a genuinely negative OOS edge. The per-window table shows the loss is a fat
# negative tail (worst_1pct_bps) on a low-breadth book, not a broad grind.
if has_risk_cols:
    g = returns['gross_exposure'].replace(0, np.nan)
    cum_pu = (1 + (returns['gross_return'] / g).fillna(0)).cumprod() - 1
    cum_net = (1 + returns['net_return'].fillna(0)).cumprod() - 1

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                        row_heights=[0.62, 0.38],
                        subplot_titles=('Signal-level edge (gross-normalized) vs net equity (%)',
                                        'Gross exposure'))
    fig.add_trace(go.Scatter(x=returns.index, y=cum_pu * 100, name='per-unit-gross (signal edge)',
                             line=dict(color='purple')), row=1, col=1)
    fig.add_trace(go.Scatter(x=returns.index, y=cum_net * 100, name='net (as traded)',
                             line=dict(color='navy')), row=1, col=1)
    fig.add_trace(go.Scatter(x=returns.index, y=returns['gross_exposure'], name='gross',
                             fill='tozeroy', line=dict(color='darkgreen')), row=2, col=1)
    fig.update_layout(height=560, template='plotly_white', hovermode='x unified')
    fig.show()

    edges = list(windows['test_start']) + [windows['test_end'].iloc[-1]]
    diag = []
    for i, wr in windows.iterrows():
        seg = returns.loc[(returns.index >= edges[i]) & (returns.index < edges[i + 1])]
        nser = seg['net_return'].dropna()
        if nser.empty:
            continue
        pu = (seg['gross_return'] / seg['gross_exposure'].replace(0, np.nan)).dropna()
        k = max(1, len(nser) // 100)
        diag.append({
            'window': int(wr['window_idx']),
            'net_%': ((1 + nser).prod() - 1) * 100,
            'signal_edge_pu_%': pu.sum() * 100,           # per-unit-gross, exposure-stripped
            'avg_gross': seg['gross_exposure'].mean(),
            'pct_neg_bars': (nser < 0).mean() * 100,
            'worst_bar_bps': nser.min() * 1e4,
            'worst_1pct_bps': nser.nsmallest(k).sum() * 1e4,
        })
    display(pd.DataFrame(diag).set_index('window').round(2))
else:
    print('Per-unit-gross diagnostics need per-bar gross_return/gross_exposure. Re-run walk_forward.py.')

## Calendar & Window Performance

In [ ]:
# Net return by WALK-FORWARD WINDOW - the unit the strategy actually retrains on.
# (Calendar-month bars were dropped: windows are ~30d and overlap months, so they
# were nearly identical to this. Short W## labels stay horizontal; dates on hover.)
edges = list(windows['test_start']) + [windows['test_end'].iloc[-1]]
win_ret = []
for i, wr in windows.iterrows():
    seg = returns.loc[(returns.index >= edges[i]) & (returns.index < edges[i + 1]), 'net_return']
    win_ret.append({
        'window': int(wr['window_idx']),
        'label': f"W{int(wr['window_idx']):02d}",
        'period': f"{pd.Timestamp(edges[i]).date()} → {pd.Timestamp(edges[i + 1]).date()}",
        'ret_%': (((1 + seg).prod() - 1) * 100) if len(seg) else np.nan,
    })
win_ret = pd.DataFrame(win_ret)

fig = go.Figure(go.Bar(
    x=win_ret['label'], y=win_ret['ret_%'],
    marker_color=['seagreen' if v > 0 else 'indianred' for v in win_ret['ret_%']],
    customdata=win_ret['period'],
    hovertemplate='%{x}  %{customdata}<br>net %{y:.2f}%<extra></extra>',
))
fig.add_hline(y=0, line_color='gray', line_width=1)
fig.update_xaxes(tickangle=0)
fig.update_layout(height=420, template='plotly_white', showlegend=False,
                  title='Net return by walk-forward window',
                  xaxis_title='window', yaxis_title='return (%)')
fig.show()

win_ret.set_index('window')[['period', 'ret_%']].round(3)

## Walk-Forward Window Diagnostics

In [ ]:
w = windows.copy()
if 'n_selected' not in w.columns or w['n_selected'].isna().all():
    def _n_selected(s):
        if not isinstance(s, str):
            return 0
        n = 0
        for part in s.split(';'):
            if ':' in part:
                n += len([x for x in part.split(':', 1)[1].split(',') if x])
        return n
    w['n_selected'] = w['selected'].apply(_n_selected)

rows = 5 if 'avg_gross' in w.columns else 4
titles = ['OOS Sharpe', 'Selected signals', 'Selection funnel', 'Train-to-OOS IC diagnostics']
if rows == 5:
    titles.insert(2, 'Average gross exposure')

fig = make_subplots(rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=titles)
fig.add_trace(go.Scatter(x=w['test_start'], y=w['oos_sharpe'], mode='lines+markers', name='OOS Sharpe'), row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_trace(go.Scatter(x=w['test_start'], y=w['n_selected'], mode='lines+markers', name='n selected'), row=2, col=1)
next_row = 3
if rows == 5:
    fig.add_trace(go.Scatter(x=w['test_start'], y=w['avg_gross'], mode='lines+markers', name='avg gross'), row=3, col=1)
    next_row = 4
for col in ['n_candidates', 'n_after_fdr', 'n_after_gates', 'n_after_bootstrap']:
    if col in w.columns:
        fig.add_trace(go.Scatter(x=w['test_start'], y=w[col], mode='lines+markers', name=col), row=next_row, col=1)
diag_row = next_row + 1
for col in ['train_oos_ic_rank_corr', 'train_oos_ic_sign_accuracy']:
    if col in w.columns:
        fig.add_trace(go.Scatter(x=w['test_start'], y=w[col], mode='lines+markers', name=col), row=diag_row, col=1)
fig.update_layout(height=850, template='plotly_white', hovermode='x unified')
fig.show()

cols = [c for c in ['window_idx', 'train_end', 'test_end', 'n_candidates', 'n_after_fdr',
                    'n_after_gates', 'n_after_bootstrap', 'n_selected', 'oos_sharpe',
                    'avg_gross', 'avg_turnover', 'avg_abs_mkt_exposure', 'avg_abs_size_exposure',
                    'train_oos_ic_rank_corr', 'train_oos_ic_sign_accuracy'] if c in w.columns]
w[cols].round(4)

## Per-Signal OOS Attribution (Option A)

Traces each window's PnL to the **individual signals** that were selected, comparing the training IC that earned a signal its slot against its realized out-of-sample IC and standalone dollar-neutral return (both in the traded direction — negative `oos_ic` means the signal flipped sign OOS). This is what turns "the first month dropped" into "*these* signals decayed/flipped" so you can drop or cap them. Requires a run saved by the updated `walk_forward.py` (`wf_portfolio_signal_attribution` table).

In [ ]:
attr = load_data('wf_portfolio_signal_attribution')
if attr is None or attr.empty:
    print('No wf_portfolio_signal_attribution table yet.\n'
          'Re-run `uv run python research/portfolio/walk_forward.py` to populate '
          'per-signal OOS attribution (Option A).')
else:
    attr = attr.copy()
    attr['oos_ret_pct'] = attr['oos_ret'] * 100

    # 1) Drill into the worst window: which signals drove its loss. train_ic and
    #    oos_ic are in the TRADED direction (positive = still working); oos_ic < 0
    #    means the signal flipped sign out-of-sample.
    worst_win = int(attr.groupby('window_idx')['oos_ret'].mean().idxmin())
    wsl = attr[attr['window_idx'] == worst_win].sort_values('oos_ret')
    print(f"Worst window = W{worst_win:02d}  "
          f"(mean signal OOS return {wsl['oos_ret'].mean() * 100:.2f}%, "
          f"{(wsl['oos_ic'] < 0).mean() * 100:.0f}% of signals flipped sign OOS)")
    display(wsl[['bucket', 'signal_name', 'family', 'sign', 'holding_lag',
                 'train_ic', 'oos_ic', 'oos_ret_pct', 'oos_sharpe']].round(4).reset_index(drop=True))

    # 2) Persistent offenders: worst standalone OOS edge across every window picked.
    by_sig = attr.groupby('signal_name').agg(
        n_sel=('oos_ret', 'size'),
        mean_oos_ret_pct=('oos_ret_pct', 'mean'),
        mean_train_ic=('train_ic', 'mean'),
        mean_oos_ic=('oos_ic', 'mean'),
        flip_rate=('oos_ic', lambda s: (s < 0).mean()),
    ).sort_values('mean_oos_ret_pct')
    print('\nWorst 15 signals by mean standalone OOS return (across windows selected):')
    display(by_sig.head(15).round(4))

    # 3) Family x window heatmap of standalone OOS return - where/when edge breaks.
    fam = attr.pivot_table(index='family', columns='window_idx', values='oos_ret_pct', aggfunc='mean')
    fig = px.imshow(fam, aspect='auto', color_continuous_scale='RdBu', zmin=-4, zmax=4,
                    origin='lower', labels=dict(color='OOS ret %', x='window', y='family'),
                    title='Standalone OOS return (%) by signal family x window')
    fig.update_layout(height=480, template='plotly_white')
    fig.show()

## Execution, Cost, and Gross Diagnostics

In [ ]:
if not has_risk_cols:
    print('Per-bar risk columns are not in this saved run. Re-run research/portfolio/walk_forward.py to populate them.')
else:
    cost = returns['gross_return'] - returns['net_return']
    ex_stats = pd.Series({
        'avg_gross': returns['gross_exposure'].mean(),
        'p95_gross': returns['gross_exposure'].quantile(0.95),
        'max_gross': returns['gross_exposure'].max(),
        'avg_turnover_bar': returns['turnover'].mean(),
        'ann_turnover_x_gross': returns['turnover'].mean() * PPY,
        'avg_cost_bps_bar': cost.mean() * 10000,
        'ann_cost_%': cost.mean() * PPY * 100,
        'avg_n_positions': returns['n_positions'].mean(),
        'min_n_positions': returns['n_positions'].min(),
        'max_n_positions': returns['n_positions'].max(),
    })
    display(ex_stats.to_frame('value').round(6))

    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=('Gross exposure', 'Turnover / bar', 'Cost drag / bar (bps)', 'Positions'))
    fig.add_trace(go.Scatter(x=returns.index, y=returns['gross_exposure'], name='gross', line=dict(color='darkgreen')), row=1, col=1)
    fig.add_trace(go.Scatter(x=returns.index, y=returns['turnover'], name='turnover', line=dict(color='orange')), row=2, col=1)
    fig.add_trace(go.Scatter(x=returns.index, y=cost * 10000, name='cost bps', line=dict(color='firebrick')), row=3, col=1)
    fig.add_trace(go.Scatter(x=returns.index, y=returns['n_positions'], name='n positions', line=dict(color='purple')), row=4, col=1)
    fig.update_layout(height=760, template='plotly_white', hovermode='x unified')
    fig.show()

## Factor Neutrality and Approximate Factor Attribution

The exposure plot is exact for the held book. The factor attribution below is approximate: the portfolio earns the next raw bar, so exposures at `t` are paired with factor returns at `t+1`.

In [ ]:
# TRUE ex-post factor attribution. The model's exposures (mkt_exposure,
# size_exposure) are CONSTRAINED to ~0 by the MVO neutrality, so multiplying them
# by factor returns is tautologically zero. Instead regress realized gross returns
# on realized factor returns to recover the book's actual (ex-post) factor betas -
# this is what catches hidden exposure from stale/misestimated loadings.
if not has_risk_cols:
    print('Realized attribution requires per-bar gross_return. Re-run walk_forward.py.')
else:
    factors = load_data('risk_factors')
    factors = factors.assign(timestamp=pd.to_datetime(factors['timestamp'])).set_index('timestamp').sort_index()
    fwd_fac = factors[['market_factor', 'size_factor']].shift(-1).reindex(returns.index)
    df = pd.DataFrame({'gross': returns['gross_return'],
                       'mkt': fwd_fac['market_factor'],
                       'size': fwd_fac['size_factor']}).dropna()
    X = np.column_stack([np.ones(len(df)), df['mkt'].values, df['size'].values])
    beta, *_ = np.linalg.lstsq(X, df['gross'].values, rcond=None)
    resid = df['gross'].values - X @ beta
    r2 = 1 - (resid ** 2).sum() / (((df['gross'] - df['gross'].mean()) ** 2).sum())

    mkt_pnl = pd.Series(beta[1] * df['mkt'].values, index=df.index)
    size_pnl = pd.Series(beta[2] * df['size'].values, index=df.index)
    alpha_pnl = pd.Series(beta[0] + resid, index=df.index)   # intercept + residual = true non-factor PnL

    display(pd.Series({
        'realized_beta_market': beta[1],
        'realized_beta_size': beta[2],
        'R2_explained_by_factors': r2,
        'market_pnl_ann_%': mkt_pnl.mean() * PPY * 100,
        'size_pnl_ann_%': size_pnl.mean() * PPY * 100,
        'alpha_pnl_ann_%': alpha_pnl.mean() * PPY * 100,
        'total_gross_ann_%': df['gross'].mean() * PPY * 100,
    }).to_frame('value').round(4))

    cum = pd.DataFrame({
        'systematic (market+size)': (mkt_pnl + size_pnl).reindex(returns.index).fillna(0.0),
        'alpha (residual)': alpha_pnl.reindex(returns.index).fillna(0.0),
        'net (after cost)': returns['net_return'].fillna(0.0),
    })
    cum = (1 + cum).cumprod() - 1

    win = BARS_PER_DAY * 20
    roll_bmkt = df['gross'].rolling(win).cov(df['mkt']) / df['mkt'].rolling(win).var()
    roll_bsize = df['gross'].rolling(win).cov(df['size']) / df['size'].rolling(win).var()

    fig = make_subplots(rows=2, cols=1, vertical_spacing=0.11, row_heights=[0.6, 0.4],
                        subplot_titles=('Cumulative PnL decomposition (%) - systematic vs alpha vs net',
                                        'Rolling 20d realized factor beta (true ex-post exposure)'))
    for c, col in zip(cum.columns, ['firebrick', 'navy', 'seagreen']):
        fig.add_trace(go.Scatter(x=cum.index, y=cum[c] * 100, name=c, line=dict(color=col)), row=1, col=1)
    fig.add_trace(go.Scatter(x=roll_bmkt.index, y=roll_bmkt, name='beta market', line=dict(color='crimson')), row=2, col=1)
    fig.add_trace(go.Scatter(x=roll_bsize.index, y=roll_bsize, name='beta size', line=dict(color='steelblue')), row=2, col=1)
    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)
    fig.update_layout(height=720, template='plotly_white', hovermode='x unified')
    fig.show()

## Return Distribution and Tail Diagnostics

In [ ]:
ret = returns['net_return'].dropna()
q = ret.quantile([0.001, 0.005, 0.01, 0.05, 0.50, 0.95, 0.99, 0.995, 0.999])
display((q * 10000).to_frame('net_return_bps').round(3))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Net return distribution (bps)', 'QQ vs normal'))
fig.add_trace(go.Histogram(x=ret * 10000, nbinsx=120, name='net bps'), row=1, col=1)
p = (np.arange(1, len(ret) + 1) - 0.5) / len(ret)
sample = np.sort((ret - ret.mean()) / ret.std())
normal = norm.ppf(p)
fig.add_trace(go.Scatter(x=normal, y=sample, mode='markers', marker=dict(size=3), name='QQ'), row=1, col=2)
lo, hi = np.nanpercentile(np.r_[normal, sample], [1, 99])
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines', line=dict(color='red', dash='dash'), name='normal'), row=1, col=2)
fig.update_layout(height=430, template='plotly_white', showlegend=False)
fig.show()

## Position, Risk & Covariance Diagnostics

Per-bar **held weights** (`wf_portfolio_weights`) and per-bar **predicted risk / covariance diagnostics** (`wf_portfolio_risk`) are now persisted by `walk_forward.py`. These enable exact position-cap audits, realized-vs-predicted risk, covariance-concentration tracking, and cluster-penalty diagnostics. Requires a run saved by the updated code (re-run `walk_forward.py` to populate).

In [ ]:
from config import get as _get
weights = load_data('wf_portfolio_weights')
risk = load_data('wf_portfolio_risk')
if weights is None or weights.empty or risk is None or risk.empty:
    print('No wf_portfolio_weights / wf_portfolio_risk tables yet.\n'
          'Re-run `uv run python research/portfolio/walk_forward.py` to populate '
          'per-name weights and per-bar risk diagnostics.')
else:
    cap = _get('portfolio.max_position') * _get('portfolio.gross_leverage')
    risk = (risk.assign(timestamp=pd.to_datetime(risk['timestamp']))
            .sort_values(['window', 'timestamp']).drop_duplicates('timestamp', keep='first')
            .set_index('timestamp').sort_index())
    weights = (weights.assign(timestamp=pd.to_datetime(weights['timestamp']))
               .sort_values(['window', 'timestamp'])
               .drop_duplicates(['timestamp', 'symbol'], keep='first'))

    # 1) Realized vs predicted vol, risk concentration, names pinned at the cap.
    realized_vol = returns['net_return'].rolling(BARS_PER_DAY * 5).std() * ANN
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=('Predicted vs realized annual vol (%)',
                                        'Covariance top-eigenvalue share (risk concentration)',
                                        'Names sitting at the position cap'))
    fig.add_trace(go.Scatter(x=risk.index, y=risk['pred_vol_ann'] * 100, name='predicted',
                             line=dict(color='purple')), row=1, col=1)
    fig.add_trace(go.Scatter(x=realized_vol.index, y=realized_vol * 100, name='realized (5d)',
                             line=dict(color='navy')), row=1, col=1)
    fig.add_trace(go.Scatter(x=risk.index, y=risk['cov_eig_top_share'], name='lambda1 share',
                             line=dict(color='teal')), row=2, col=1)
    fig.add_trace(go.Scatter(x=risk.index, y=risk['n_at_cap'], name='n at cap',
                             line=dict(color='firebrick')), row=3, col=1)
    fig.update_layout(height=720, template='plotly_white', hovermode='x unified', showlegend=True)
    fig.show()

    # 2) Position-cap audit + the names that carry the most weight on average.
    pct_at_cap = (risk['n_at_cap'] > 0).mean() * 100
    print(f"position cap = {cap:.4f} | max |weight| ever = {weights['weight'].abs().max():.4f} | "
          f"bars with >=1 name at the cap: {pct_at_cap:.1f}%")
    top = (weights.assign(aw=weights['weight'].abs())
           .groupby('symbol').agg(mean_abs_w=('aw', 'mean'), max_abs_w=('aw', 'max'),
                                  bars_held=('aw', 'size'))
           .sort_values('mean_abs_w', ascending=False).head(15))
    display(top.round(4))

## Realized OOS IC vs Selection `IC_rho`

`IC_rho` and `sign` in the window table are **persistence** diagnostics: the
rank / sign agreement between each selected signal's *train* IC and its *OOS* IC,
across the 2-10 selected signals (`persistence_diagnostic` in `walk_forward.py`).
They say the picks kept pointing the same way - nothing about **magnitude** or
**PnL**. This cell plots the **realized OOS IC actually earned** (mean across the
window's selected signals, traded direction, from
`wf_portfolio_signal_attribution`) next to `IC_rho` and reports how each
correlates with the window's OOS Sharpe. Read `IC_rho` against the realized OOS
IC and the Sharpe, never on its own: the question is whether the selected edge
(a) survives out-of-sample and (b) converts into portfolio PnL. A positive
realized OOS IC that fails to lift the Sharpe points at portfolio
construction / costs / breadth rather than signal decay - which is what the
equal-weight-vs-MVO and family/breadth cells below probe.

In [ ]:
attr = load_data('wf_portfolio_signal_attribution')
if attr is None or attr.empty:
    print('No wf_portfolio_signal_attribution table. Re-run research/portfolio/walk_forward.py.')
else:
    # Realized OOS IC earned, averaged over the window's selected signals in
    # their traded direction (oos_ic is already sign-applied in attribution).
    wic = (attr.groupby('window_idx')
                .agg(oos_ic=('oos_ic', 'mean'),
                     train_ic=('train_ic', 'mean'),
                     n_sel=('signal_name', 'size')).reset_index())
    w = (windows[['window_idx', 'train_oos_ic_rank_corr', 'oos_sharpe']]
         .merge(wic, on='window_idx', how='left').sort_values('window_idx'))
    lab = [f"W{int(i):02d}" for i in w['window_idx']]

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_bar(x=lab, y=w['oos_ic'], name='realized OOS IC (selected, traded dir)',
                marker_color=['seagreen' if v > 0 else 'indianred'
                              for v in w['oos_ic'].fillna(0)])
    fig.add_trace(go.Scatter(x=lab, y=w['train_ic'], name='train IC (earned the slot)',
                             mode='lines+markers',
                             line=dict(color='goldenrod', dash='dot')), secondary_y=False)
    fig.add_trace(go.Scatter(x=lab, y=w['train_oos_ic_rank_corr'],
                             name='IC_rho (persistence, RHS)', mode='lines+markers',
                             line=dict(color='slateblue')), secondary_y=True)
    fig.add_hline(y=0, line_color='gray', line_width=1)
    fig.update_yaxes(title_text='IC (cross-sectional)', secondary_y=False)
    fig.update_yaxes(title_text='IC_rho (train->oos rank corr)', secondary_y=True, range=[-1, 1])
    fig.update_layout(height=460, template='plotly_white',
                      title='Realized OOS IC vs selection persistence (IC_rho) by window',
                      legend=dict(orientation='h', y=-0.25))
    fig.show()

    corr = w[['oos_ic', 'oos_sharpe']].corr().iloc[0, 1]
    rho_corr = w[['train_oos_ic_rank_corr', 'oos_sharpe']].corr().iloc[0, 1]
    print(f"mean realized OOS IC : {w['oos_ic'].mean():+.4f}   "
          f"(mean train IC {w['train_ic'].mean():+.4f}; "
          f"OOS retains {w['oos_ic'].mean() / w['train_ic'].mean() * 100:.0f}% of train IC)")
    print(f"corr(realized OOS IC, OOS Sharpe) : {corr:+.2f}")
    print(f"corr(IC_rho,          OOS Sharpe) : {rho_corr:+.2f}")
    print('Read together: if realized OOS IC is positive but the Sharpe is not, '
          'the leak is in portfolio construction / costs / breadth, not signal\n'
          'decay - see the equal-weight-vs-MVO and family/breadth cells.')
    display(w.assign(label=lab).set_index('label')[
        ['n_sel', 'train_ic', 'oos_ic', 'train_oos_ic_rank_corr', 'oos_sharpe']].round(4))

## Production Sizing (Rank / Equal-Weight) vs MVO Benchmark

The production book now sizes by the cross-sectional **rank** of alpha
(`portfolio.weight_scheme = 'equal_weight'`); the Ledoit-Wolf **MVO** runs
alongside as a monitored benchmark, persisted to `wf_portfolio_returns_bench`.
Both share selection, alpha, neutrality, cap and gross, so the gap is purely the
risk model. This cell tracks that the decision still holds on the clean,
de-overlapped basis: **production - benchmark > 0** means rank sizing is winning.
Re-run `research/portfolio/walk_forward.py` to refresh both books.

In [ ]:
bench = load_data('wf_portfolio_returns_bench')
if bench is None or bench.empty:
    print('No wf_portfolio_returns_bench table yet.\n'
          'Re-run: uv run python research/portfolio/walk_forward.py')
else:
    bench['timestamp'] = pd.to_datetime(bench['timestamp'])
    if 'window' in bench.columns:  # de-overlap window boundaries, as for production
        bench = bench.sort_values(['window', 'timestamp']).drop_duplicates('timestamp', keep='first')
    bench = bench.set_index('timestamp').sort_index()

    common = returns.index.intersection(bench.index)
    prod_r = returns.loc[common, 'net_return'].fillna(0)   # production = rank/equal-weight
    bench_r = bench.loc[common, 'net_return'].fillna(0)    # benchmark  = MVO (LW cov)

    cmp = pd.DataFrame({'Production (rank/equal-weight)': perf_stats(prod_r),
                        'Benchmark (MVO, LW cov)': perf_stats(bench_r)}).T
    display(cmp.round(3))

    prod_cum, bench_cum = (1 + prod_r).cumprod() - 1, (1 + bench_r).cumprod() - 1
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=prod_cum.index, y=prod_cum * 100,
                             name='Production (rank/equal-weight)', line=dict(color='darkorange')))
    fig.add_trace(go.Scatter(x=bench_cum.index, y=bench_cum * 100,
                             name='Benchmark (MVO, LW cov)', line=dict(color='steelblue')))
    fig.add_hline(y=0, line_color='gray', line_width=1)
    fig.update_layout(height=420, template='plotly_white',
                      title='Cumulative net return: production rank sizing vs MVO benchmark (same alpha/selection)',
                      yaxis_title='cumulative net return (%)',
                      legend=dict(orientation='h', y=-0.2))
    fig.show()

    rows = []
    for i, wr in windows.iterrows():
        s, e = edges[i], edges[i + 1]
        seg_p = prod_r[(prod_r.index >= s) & (prod_r.index < e)]
        seg_b = bench_r[(bench_r.index >= s) & (bench_r.index < e)]
        rows.append({'label': f"W{int(wr['window_idx']):02d}",
                     'Prod_%': ((1 + seg_p).prod() - 1) * 100 if len(seg_p) else np.nan,
                     'MVO_%': ((1 + seg_b).prod() - 1) * 100 if len(seg_b) else np.nan})
    cw = pd.DataFrame(rows)
    fig2 = go.Figure()
    fig2.add_bar(x=cw['label'], y=cw['Prod_%'], name='Production (rank)', marker_color='darkorange')
    fig2.add_bar(x=cw['label'], y=cw['MVO_%'], name='MVO benchmark', marker_color='steelblue')
    fig2.add_hline(y=0, line_color='gray', line_width=1)
    fig2.update_layout(barmode='group', height=420, template='plotly_white',
                       title='Net return by window: production rank vs MVO benchmark',
                       yaxis_title='return (%)', legend=dict(orientation='h', y=-0.2))
    fig2.show()

    d = prod_r - bench_r
    print(f"Production (rank) Sharpe {perf_stats(prod_r)['sharpe']:.2f}  vs  "
          f"MVO benchmark Sharpe {perf_stats(bench_r)['sharpe']:.2f}")
    print(f"production-minus-benchmark mean bps/bar: {d.mean() * 1e4:+.3f}  "
          f"(>0 => rank sizing beats the covariance MVO)")

## Family Concentration & Tail (Q5-Q1) Decomposition

Two structural risks flagged in the analysis: (1) the book is dominated by one
mean-reverting family (`residual_reversion`), and (2) reversion signals tend to
have a high hit-rate but **negative skew** - they win small and lose big, so a
positive average IC need not be profitable once the MVO sizes the tails. Below:
family selection frequency, realized OOS edge per family, and the **Q5-Q1 tail
spread** (`q_spread` from `signal_metrics`) of the selected signals - whether the
alpha actually lives in the tradeable tails. Portfolio-level return skew is in
the tail-diagnostics cell above.

In [ ]:
attr = load_data('wf_portfolio_signal_attribution')
metrics = load_data('signal_metrics')
if attr is None or attr.empty:
    print('No wf_portfolio_signal_attribution table. Re-run research/portfolio/walk_forward.py.')
else:
    famcount = attr['family'].value_counts()
    famedge = attr.groupby('family').agg(
        picks=('signal_name', 'size'),
        oos_ic=('oos_ic', 'mean'),
        oos_sharpe=('oos_sharpe', 'median'),
        oos_ret_bps=('oos_ret', lambda s: np.nanmean(s) * 1e4),
    ).sort_values('picks', ascending=False)
    display(famedge.round(4))

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        'Family selection frequency (all windows)',
        'Realized OOS IC by family (traded direction)'))
    fig.add_bar(x=famcount.index, y=famcount.values, marker_color='slateblue',
                row=1, col=1, showlegend=False)
    fe = famedge.reset_index()
    fig.add_bar(x=fe['family'], y=fe['oos_ic'], row=1, col=2, showlegend=False,
                marker_color=['seagreen' if v > 0 else 'indianred' for v in fe['oos_ic']])
    fig.add_hline(y=0, line_color='gray', line_width=1, row=1, col=2)
    fig.update_xaxes(tickangle=-40)
    fig.update_layout(height=440, template='plotly_white',
                      title='Family concentration and realized OOS edge')
    fig.show()

    # Q5-Q1 tail spread of SELECTED signals, by family. q_spread = mean
    # top-quintile minus bottom-quintile forward residual return per
    # cross-section: the alpha must live in these tradeable tails, not in a
    # monotone-but-flat middle that rank IC alone would still reward.
    if metrics is not None and not metrics.empty:
        fam_map = attr.drop_duplicates('signal_name').set_index('signal_name')['family']
        mm = metrics[metrics['signal_name'].isin(set(attr['signal_name']))].copy()
        mm['family'] = mm['signal_name'].map(fam_map)
        qs = mm.groupby(['family', 'signal_name'], as_index=False)['q_spread'].mean()
        fig3 = px.box(qs, x='family', y='q_spread', points='all', color='family',
                      title='Q5-Q1 tail spread of selected signals, by family')
        fig3.add_hline(y=0, line_color='gray', line_width=1)
        fig3.update_xaxes(tickangle=-40)
        fig3.update_layout(height=460, template='plotly_white', showlegend=False)
        fig3.show()
        rr = qs[qs['family'].str.contains('revers', case=False, na=False)]
        if not rr.empty:
            print(f"residual_reversion selected signals: {len(rr)}   "
                  f"median Q5-Q1 spread {rr['q_spread'].median() * 1e4:+.2f} bps")
    else:
        print('signal_metrics empty - run research/signals/evaluate.py for q_spread.')